# NYC Yellow Taxi Operations & Revenue Analysis

## Data Understanding and Data Quality Audit

This notebook documents the initial inspection and quality assessment of the January 2024 NYC Yellow Taxi dataset.

### Purpose

- understand the structure and grain of the source tables;
- inspect data types, date ranges, and categorical values;
- identify missing values, invalid records, and outliers;
- verify duplicate records and taxi-zone relationships;
- define metric-specific treatment rules before loading the data into PostgreSQL.

### Project approach

This is a **SQL-first portfolio project**. Python is used here only for source-file inspection and data-quality validation. The main business analysis will be performed in PostgreSQL.


## 1. Load source files

Upload the January 2024 Yellow Taxi Parquet file and the taxi-zone lookup CSV file.

The code detects the uploaded files by extension, so it also works when Colab adds a suffix such as `(1)` to a duplicate filename.


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
uploaded_files = list(uploaded.keys())

trip_file = next(
    file_name for file_name in uploaded_files
    if file_name.lower().endswith(".parquet")
)

zone_file = next(
    file_name for file_name in uploaded_files
    if file_name.lower().endswith(".csv")
)

trips = pd.read_parquet(trip_file)
zones = pd.read_csv(zone_file)

print("Trip file:", trip_file)
print("Zone file:", zone_file)
print("Trips shape:", trips.shape)
print("Zones shape:", zones.shape)

print("\nTrip columns:")
print(trips.columns.tolist())


## 2. Table structure and data types

Inspect row counts, non-null counts, column names, and pandas data types for both source tables.


In [3]:
print("TRIPS TABLE INFO")
print("-" * 50)
trips.info(show_counts=True)

print("\nZONES TABLE INFO")
print("-" * 50)
zones.info(show_counts=True)

TRIPS TABLE INFO
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2964624 entries, 0 to 2964623
Data columns (total 19 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               2964624 non-null  int32         
 1   tpep_pickup_datetime   2964624 non-null  datetime64[us]
 2   tpep_dropoff_datetime  2964624 non-null  datetime64[us]
 3   passenger_count        2824462 non-null  float64       
 4   trip_distance          2964624 non-null  float64       
 5   RatecodeID             2824462 non-null  float64       
 6   store_and_fwd_flag     2824462 non-null  object        
 7   PULocationID           2964624 non-null  int32         
 8   DOLocationID           2964624 non-null  int32         
 9   payment_type           2964624 non-null  int64         
 10  fare_amount            2964624 non-null  float64       
 11  extra                

## 3. Date ranges and categorical values

Confirm the actual pickup/drop-off period and inspect the distribution of key categorical fields.


In [4]:
# Проверяем фактический период поездок.
print("Pickup date range:")
print("From:", trips["tpep_pickup_datetime"].min())
print("To:  ", trips["tpep_pickup_datetime"].max())

print("\nDrop-off date range:")
print("From:", trips["tpep_dropoff_datetime"].min())
print("To:  ", trips["tpep_dropoff_datetime"].max())

# Просматриваем возможные значения основных категориальных колонок.
categorical_columns = [
    "VendorID",
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "payment_type"
]

for column in categorical_columns:
    print(f"\nValues in {column}:")
    print(
        trips[column]
        .value_counts(dropna=False)
        .sort_index()
    )

Pickup date range:
From: 2002-12-31 22:59:39
To:   2024-02-01 00:01:15

Drop-off date range:
From: 2002-12-31 23:05:41
To:   2024-02-02 13:56:52

Values in VendorID:
VendorID
1     729732
2    2234632
6        260
Name: count, dtype: int64

Values in passenger_count:
passenger_count
0.0      31465
1.0    2188739
2.0     405103
3.0      91262
4.0      51974
5.0      33506
6.0      22353
7.0          8
8.0         51
9.0          1
NaN     140162
Name: count, dtype: int64

Values in RatecodeID:
RatecodeID
1.0     2663350
2.0       98713
3.0        7954
4.0        6365
5.0       19410
6.0           7
99.0      28663
NaN      140162
Name: count, dtype: int64

Values in store_and_fwd_flag:
store_and_fwd_flag
N       2813126
Y         11336
None     140162
Name: count, dtype: int64

Values in payment_type:
payment_type
0     140162
1    2319046
2     439191
3      19597
4      46628
Name: count, dtype: int64


## 4. Descriptive statistics

Review numeric, text, and datetime summaries to identify unusual ranges and possible anomalies.


In [5]:
print("=== Numeric columns ===")
display(trips.describe().T)

print("\n=== Object columns ===")
display(trips.select_dtypes(include="object").describe().T)

print("\n=== Datetime columns ===")
display(trips.select_dtypes(include="datetime").describe().T)

=== Numeric columns ===


,count,mean,min,25%,50%,75%,max,std
VendorID,2964624.0,1.754204,1.0,2.0,2.0,2.0,6.0,0.43259
tpep_pickup_datetime,2964624,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15,NaN
tpep_dropoff_datetime,2964624,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52,NaN
passenger_count,2824462.0,1.339281,0.0,1.0,1.0,1.0,9.0,0.850282
trip_distance,2964624.0,3.652169,0.0,1.0,1.68,3.11,312722.3,225.462572
RatecodeID,2824462.0,2.069359,1.0,1.0,1.0,1.0,99.0,9.823219
PULocationID,2964624.0,166.017884,1.0,132.0,162.0,234.0,265.0,63.623914
DOLocationID,2964624.0,165.116712,1.0,114.0,162.0,234.0,265.0,69.31535
payment_type,2964624.0,1.161271,0.0,1.0,1.0,1.0,4.0,0.580869
fare_amount,2964624.0,18.175062,-899.0,8.6,12.8,20.5,5000.0,18.949548



=== Object columns ===


,count,unique,top,freq
store_and_fwd_flag,2824462,2,N,2813126



=== Datetime columns ===


,count,mean,min,25%,50%,75%,max
tpep_pickup_datetime,2964624,2024-01-17 00:46:36.431092,2002-12-31 22:59:39,2024-01-09 15:59:19.750000,2024-01-17 10:45:37.500000,2024-01-24 18:23:52.250000,2024-02-01 00:01:15
tpep_dropoff_datetime,2964624,2024-01-17 01:02:13.208130,2002-12-31 23:05:41,2024-01-09 16:16:23,2024-01-17 11:03:51.500000,2024-01-24 18:40:29,2024-02-02 13:56:52


## 5. Column-level value inspection

Display representative values for every column. This supports the data dictionary and helps distinguish identifiers, categories, measures, and timestamps.


In [6]:
# Показываем пример значений для каждой колонки

for column in trips.columns:
    print("=" * 70)
    print(f"Column: {column}")
    print(f"Data type: {trips[column].dtype}")

    if trips[column].dtype == "object":
        print("\nSample values:")
        print(trips[column].dropna().unique()[:10])

    elif "datetime" in str(trips[column].dtype):
        print("\nFirst 5 values:")
        print(trips[column].head())

    else:
        print("\nUnique values (up to 10):")
        print(trips[column].dropna().unique()[:10])

Column: VendorID
Data type: int32

Unique values (up to 10):
[2 1 6]
Column: tpep_pickup_datetime
Data type: datetime64[us]

First 5 values:
0   2024-01-01 00:57:55
1   2024-01-01 00:03:00
2   2024-01-01 00:17:06
3   2024-01-01 00:36:38
4   2024-01-01 00:46:51
Name: tpep_pickup_datetime, dtype: datetime64[us]
Column: tpep_dropoff_datetime
Data type: datetime64[us]

First 5 values:
0   2024-01-01 01:17:43
1   2024-01-01 00:09:36
2   2024-01-01 00:35:01
3   2024-01-01 00:44:56
4   2024-01-01 00:52:57
Name: tpep_dropoff_datetime, dtype: datetime64[us]
Column: passenger_count
Data type: float64

Unique values (up to 10):
[1. 2. 0. 4. 3. 5. 6. 8. 7. 9.]
Column: trip_distance
Data type: float64

Unique values (up to 10):
[ 1.72  1.8   4.7   1.4   0.8  10.82  3.    5.44  0.04  0.75]
Column: RatecodeID
Data type: float64

Unique values (up to 10):
[ 1.  5.  2.  4. 99.  3.  6.]
Column: store_and_fwd_flag
Data type: object

Sample values:
['N' 'Y']
Column: PULocationID
Data type: int32

Unique v

## 6. Missing-value summary

Calculate the number and percentage of missing values for every trip column.


In [7]:
import pandas as pd

quality_summary = pd.DataFrame({
    "Column": trips.columns,
    "Data Type": trips.dtypes.astype(str).values,
    "Non-Null": trips.notnull().sum().values,
    "Missing": trips.isnull().sum().values,
})

quality_summary["Missing %"] = (
    quality_summary["Missing"] / len(trips) * 100
).round(2)

quality_summary

,Column,Data Type,Non-Null,Missing,Missing %
0,VendorID,int32,2964624,0,0.00
1,tpep_pickup_datetime,datetime64[us],2964624,0,0.00
2,tpep_dropoff_datetime,datetime64[us],2964624,0,0.00
3,passenger_count,float64,2824462,140162,4.73
4,trip_distance,float64,2964624,0,0.00
5,RatecodeID,float64,2824462,140162,4.73
6,store_and_fwd_flag,object,2824462,140162,4.73
7,PULocationID,int32,2964624,0,0.00
8,DOLocationID,int32,2964624,0,0.00
9,payment_type,int64,2964624,0,0.00


## 7. Missing-value pattern consistency

Check whether the five columns with missing data are absent in exactly the same records.


In [8]:
# Проверяем, совпадают ли пропуски в разных колонках.

missing_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee"
]

missing_mask = trips[missing_columns].isnull()

print("Rows where ALL five columns are missing:")
print(missing_mask.all(axis=1).sum())

print("\nRows where AT LEAST ONE column is missing:")
print(missing_mask.any(axis=1).sum())

print("\nRows where missing pattern is inconsistent:")
print(
    missing_mask.any(axis=1).sum()
    - missing_mask.all(axis=1).sum()
)

Rows where ALL five columns are missing:
140162

Rows where AT LEAST ONE column is missing:
140162

Rows where missing pattern is inconsistent:
0


## 8. Profile the missing-value group

Investigate vendor, payment type, date coverage, and usable fields in the 140,162-row missing-value group.


In [9]:
# Исследуем строки с одинаковым шаблоном пропусков

missing_group = trips[trips["passenger_count"].isna()]

print("Rows in missing group:", len(missing_group))

print("\nVendorID:")
print(missing_group["VendorID"].value_counts(dropna=False))

print("\nPayment Type:")
print(missing_group["payment_type"].value_counts(dropna=False))

print("\nPickup date range:")
print(missing_group["tpep_pickup_datetime"].min())
print(missing_group["tpep_pickup_datetime"].max())

print("\nSample rows:")
display(
    missing_group[
        [
            "VendorID",
            "payment_type",
            "trip_distance",
            "fare_amount",
            "total_amount",
            "PULocationID",
            "DOLocationID"
        ]
    ].head(10)
)

Rows in missing group: 140162

VendorID:
VendorID
2    91447
1    48455
6      260
Name: count, dtype: int64

Payment Type:
payment_type
0    140162
Name: count, dtype: int64

Pickup date range:
2024-01-01 00:00:58
2024-01-31 23:58:25

Sample rows:


,VendorID,payment_type,trip_distance,fare_amount,total_amount,PULocationID,DOLocationID
2824462,2,0,2.04,12.72,16.72,143,141
2824463,1,0,1.60,9.30,17.16,236,238
2824464,1,0,0.00,21.01,25.01,142,79
2824465,1,0,0.00,17.79,21.79,237,4
2824466,1,0,0.00,34.65,38.65,244,50
2824467,2,0,4.58,28.14,29.64,202,83
2824468,2,0,1.47,12.82,16.82,224,144
2824469,1,0,1.70,10.16,11.66,79,249
2824470,2,0,4.80,33.12,44.54,158,239
2824471,2,0,1.27,11.74,18.89,211,90


## 9. Validate the payment-type relationship

Confirm that the missing-value pattern matches `payment_type = 0` row by row and forms one continuous block in the source file.


In [10]:
# Маска строк с общим шаблоном пропусков
missing_group_mask = trips["passenger_count"].isna()

# Проверяем полное совпадение:
# payment_type = 0 тогда и только тогда, когда есть общий шаблон пропусков
payment_zero_mask = trips["payment_type"].eq(0)

print("Rows with missing pattern:", missing_group_mask.sum())
print("Rows with payment_type = 0:", payment_zero_mask.sum())

print(
    "Exact row-by-row match:",
    missing_group_mask.equals(payment_zero_mask)
)

# Проверяем расположение таких строк в DataFrame
missing_indices = trips.index[missing_group_mask]

print("\nFirst missing-group index:", missing_indices.min())
print("Last missing-group index: ", missing_indices.max())
print("Number of indices:        ", len(missing_indices))

is_contiguous = (
    len(missing_indices)
    == missing_indices.max() - missing_indices.min() + 1
)

print("Forms one continuous block:", is_contiguous)

# Показываем границу между обычными строками и missing group
display(
    trips.loc[
        missing_indices.min() - 3 : missing_indices.min() + 3,
        [
            "VendorID",
            "passenger_count",
            "RatecodeID",
            "store_and_fwd_flag",
            "payment_type",
            "congestion_surcharge",
            "Airport_fee",
            "trip_distance",
            "total_amount"
        ]
    ]
)

Rows with missing pattern: 140162
Rows with payment_type = 0: 140162
Exact row-by-row match: True

First missing-group index: 2824462
Last missing-group index:  2964623
Number of indices:         140162
Forms one continuous block: True


,VendorID,passenger_count,RatecodeID,store_and_fwd_flag,payment_type,congestion_surcharge,Airport_fee,trip_distance,total_amount
2824459,1,3.0,1.0,N,1,2.5,0.0,3.30,23.70
2824460,1,0.0,1.0,N,2,2.5,0.0,0.40,10.10
2824461,2,1.0,1.0,N,1,2.5,0.0,1.58,16.32
2824462,2,NaN,NaN,None,0,NaN,NaN,2.04,16.72
2824463,1,NaN,NaN,None,0,NaN,NaN,1.60,17.16
2824464,1,NaN,NaN,None,0,NaN,NaN,0.00,25.01
2824465,1,NaN,NaN,None,0,NaN,NaN,0.00,21.79


## 10. Basic validity checks

Count invalid or suspicious dates, durations, distances, fares, tips, and total amounts.


In [11]:
# Рассчитываем длительность поездки в минутах.
trip_duration_minutes = (
    trips["tpep_dropoff_datetime"]
    - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

validity_summary = pd.DataFrame({
    "Check": [
        "Duration <= 0 minutes",
        "Pickup outside January 2024",
        "Drop-off before January 2024",
        "Drop-off after January 2024",
        "Trip distance = 0",
        "Trip distance < 0",
        "Fare amount = 0",
        "Fare amount < 0",
        "Tip amount < 0",
        "Total amount = 0",
        "Total amount < 0"
    ],
    "Rows": [
        (trip_duration_minutes <= 0).sum(),

        (
            (trips["tpep_pickup_datetime"] < "2024-01-01")
            | (trips["tpep_pickup_datetime"] >= "2024-02-01")
        ).sum(),

        (
            trips["tpep_dropoff_datetime"] < "2024-01-01"
        ).sum(),

        (
            trips["tpep_dropoff_datetime"] >= "2024-02-01"
        ).sum(),

        (trips["trip_distance"] == 0).sum(),
        (trips["trip_distance"] < 0).sum(),
        (trips["fare_amount"] == 0).sum(),
        (trips["fare_amount"] < 0).sum(),
        (trips["tip_amount"] < 0).sum(),
        (trips["total_amount"] == 0).sum(),
        (trips["total_amount"] < 0).sum()
    ]
})

validity_summary["Percent"] = (
    validity_summary["Rows"] / len(trips) * 100
).round(4)

display(validity_summary)

,Check,Rows,Percent
0,Duration <= 0 minutes,870,0.0293
1,Pickup outside January 2024,18,0.0006
2,Drop-off before January 2024,8,0.0003
3,Drop-off after January 2024,603,0.0203
4,Trip distance = 0,60371,2.0364
5,Trip distance < 0,0,0.0000
6,Fare amount = 0,893,0.0301
7,Fare amount < 0,37448,1.2632
8,Tip amount < 0,102,0.0034
9,Total amount = 0,416,0.0140


## 11. Negative monetary records

Measure overlap between negative fare and total amounts and inspect their payment types and rate codes.


In [12]:
# Создаём маски для отрицательных денежных значений.
negative_fare_mask = trips["fare_amount"] < 0
negative_total_mask = trips["total_amount"] < 0
negative_tip_mask = trips["tip_amount"] < 0

print("Negative fare rows:", negative_fare_mask.sum())
print("Negative total rows:", negative_total_mask.sum())
print("Negative tip rows:", negative_tip_mask.sum())

print("\nRows where fare and total are both negative:")
print((negative_fare_mask & negative_total_mask).sum())

print("\nNegative fare but total is not negative:")
print((negative_fare_mask & ~negative_total_mask).sum())

print("\nNegative total but fare is not negative:")
print((negative_total_mask & ~negative_fare_mask).sum())

print("\nPayment types for negative total rows:")
print(
    trips.loc[negative_total_mask, "payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRate codes for negative total rows:")
print(
    trips.loc[negative_total_mask, "RatecodeID"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample negative total records:")
display(
    trips.loc[
        negative_total_mask,
        [
            "VendorID",
            "tpep_pickup_datetime",
            "payment_type",
            "RatecodeID",
            "trip_distance",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "total_amount"
        ]
    ].head(10)
)

Negative fare rows: 37448
Negative total rows: 35504
Negative tip rows: 102

Rows where fare and total are both negative:
35384

Negative fare but total is not negative:
2064

Negative total but fare is not negative:
120

Payment types for negative total rows:
payment_type
0        2
1       29
2     8326
3     5741
4    21406
Name: count, dtype: int64

Rate codes for negative total rows:
RatecodeID
1.0    32176
2.0     2048
3.0      335
4.0      220
5.0      721
6.0        2
NaN        2
Name: count, dtype: int64

Sample negative total records:


,VendorID,tpep_pickup_datetime,payment_type,RatecodeID,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
99,2,2024-01-01 00:18:24,4,1.0,2.16,-13.5,-1.0,-0.5,0.0,0.0,-18.50
506,2,2024-01-01 00:04:00,2,5.0,0.01,-31.5,0.0,0.0,0.0,0.0,-34.25
536,2,2024-01-01 00:41:42,4,1.0,0.47,-5.8,-1.0,-0.5,0.0,0.0,-10.80
552,2,2024-01-01 00:42:02,2,1.0,5.48,-33.1,-1.0,-0.5,0.0,0.0,-38.10
682,2,2024-01-01 00:24:02,4,1.0,8.74,-47.8,-1.0,-0.5,0.0,0.0,-52.80
999,2,2024-01-01 00:14:22,4,1.0,1.17,-9.3,-1.0,-0.5,0.0,0.0,-14.30
1057,2,2024-01-01 00:45:56,4,1.0,1.57,-11.4,-1.0,-0.5,0.0,0.0,-16.40
1195,2,2024-01-01 00:30:18,4,1.0,9.60,-40.1,-1.0,-0.5,0.0,0.0,-45.10
1382,2,2024-01-01 00:33:28,4,1.0,0.00,-3.0,-1.0,-0.5,0.0,0.0,-8.00
1472,2,2024-01-01 00:30:10,4,1.0,1.58,-11.4,-1.0,-0.5,0.0,0.0,-16.40


## 12. Sign-mismatch exceptions

Inspect records where the signs of `fare_amount` and `total_amount` do not match.


In [13]:
# Группа 1:
# fare отрицательный, но итоговая сумма не отрицательная
negative_fare_positive_total = trips[
    (trips["fare_amount"] < 0)
    & (trips["total_amount"] >= 0)
]

print("NEGATIVE FARE, NON-NEGATIVE TOTAL")
print("Rows:", len(negative_fare_positive_total))

print("\nPayment types:")
print(
    negative_fare_positive_total["payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample rows:")
display(
    negative_fare_positive_total[
        [
            "VendorID",
            "payment_type",
            "RatecodeID",
            "trip_distance",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "improvement_surcharge",
            "congestion_surcharge",
            "Airport_fee",
            "total_amount"
        ]
    ].head(10)
)


# Группа 2:
# total отрицательный, но fare не отрицательный
negative_total_nonnegative_fare = trips[
    (trips["total_amount"] < 0)
    & (trips["fare_amount"] >= 0)
]

print("\n" + "=" * 70)
print("NEGATIVE TOTAL, NON-NEGATIVE FARE")
print("Rows:", len(negative_total_nonnegative_fare))

print("\nPayment types:")
print(
    negative_total_nonnegative_fare["payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nSample rows:")
display(
    negative_total_nonnegative_fare[
        [
            "VendorID",
            "payment_type",
            "RatecodeID",
            "trip_distance",
            "fare_amount",
            "extra",
            "mta_tax",
            "tip_amount",
            "tolls_amount",
            "improvement_surcharge",
            "congestion_surcharge",
            "Airport_fee",
            "total_amount"
        ]
    ].head(10)
)

NEGATIVE FARE, NON-NEGATIVE TOTAL
Rows: 2064

Payment types:
payment_type
0    2064
Name: count, dtype: int64

Sample rows:


,VendorID,payment_type,RatecodeID,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee,total_amount
2824576,2,0,NaN,4.92,-0.42,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.58
2824646,2,0,NaN,1.46,-1.00,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.00
2824707,2,0,NaN,0.80,-1.00,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.00
2824832,2,0,NaN,2.33,-1.81,0.0,0.5,0.0,0.0,1.0,NaN,NaN,2.19
2824833,2,0,NaN,2.86,-1.61,0.0,0.5,0.0,0.0,1.0,NaN,NaN,2.39
2824982,2,0,NaN,2.70,-0.41,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.59
2825126,2,0,NaN,0.00,-1.00,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.00
2825193,2,0,NaN,0.96,-1.00,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.00
2825362,2,0,NaN,1.21,-1.51,0.0,0.5,0.0,0.0,1.0,NaN,NaN,2.49
2825434,2,0,NaN,3.55,-0.86,0.0,0.5,0.0,0.0,1.0,NaN,NaN,3.14



NEGATIVE TOTAL, NON-NEGATIVE FARE
Rows: 120

Payment types:
payment_type
1      1
2    118
3      1
Name: count, dtype: int64

Sample rows:


,VendorID,payment_type,RatecodeID,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee,total_amount
6955,2,2,1.0,2.97,0.0,0.0,-0.5,0.0,0.0,-1.0,-2.5,0.00,-4.00
15730,2,2,1.0,8.86,0.0,0.0,-0.5,0.0,0.0,-1.0,-2.5,0.00,-4.00
20847,2,2,5.0,0.00,0.0,0.0,-0.5,0.0,0.0,-1.0,0.0,0.00,-1.50
20849,2,2,5.0,0.00,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.00,-1.00
40343,2,2,1.0,0.00,0.0,0.0,-0.5,0.0,0.0,-1.0,0.0,0.00,-1.50
76754,2,2,1.0,0.17,0.0,0.0,-0.5,0.0,0.0,-1.0,-2.5,-1.75,-5.75
144598,2,2,1.0,0.08,0.0,0.0,-0.5,0.0,0.0,-1.0,0.0,-1.75,-3.25
177885,2,2,1.0,3.84,0.0,0.0,-0.5,0.0,0.0,-1.0,-2.5,0.00,-4.00
224416,2,2,1.0,5.29,0.0,0.0,-0.5,0.0,0.0,-1.0,-2.5,0.00,-4.00
225504,2,2,5.0,0.04,0.0,0.0,0.0,0.0,0.0,-1.0,0.0,0.00,-1.00


## 13. Zero-distance trips

Determine whether zero-distance records still contain usable time, zone, and financial information.


In [14]:
# Выбираем поездки с нулевой дистанцией.
zero_distance = trips[trips["trip_distance"] == 0].copy()

# Рассчитываем их длительность в минутах.
zero_distance["duration_minutes"] = (
    zero_distance["tpep_dropoff_datetime"]
    - zero_distance["tpep_pickup_datetime"]
).dt.total_seconds() / 60

print("Zero-distance rows:", len(zero_distance))

print("\nPayment types:")
print(
    zero_distance["payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRate codes:")
print(
    zero_distance["RatecodeID"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nFinancial status:")
print("Total amount > 0:", (zero_distance["total_amount"] > 0).sum())
print("Total amount = 0:", (zero_distance["total_amount"] == 0).sum())
print("Total amount < 0:", (zero_distance["total_amount"] < 0).sum())

print("\nDuration status:")
print("Duration > 0:", (zero_distance["duration_minutes"] > 0).sum())
print("Duration = 0:", (zero_distance["duration_minutes"] == 0).sum())
print("Duration < 0:", (zero_distance["duration_minutes"] < 0).sum())

print("\nDuration summary:")
display(
    zero_distance["duration_minutes"]
    .describe()
    .to_frame(name="duration_minutes")
)

print("\nSample zero-distance rows:")
display(
    zero_distance[
        [
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "duration_minutes",
            "payment_type",
            "RatecodeID",
            "fare_amount",
            "total_amount",
            "PULocationID",
            "DOLocationID"
        ]
    ].head(10)
)

Zero-distance rows: 60371

Payment types:
payment_type
0    22825
1    20604
2     8583
3     4566
4     3793
Name: count, dtype: int64

Rate codes:
RatecodeID
1.0     22215
2.0      3051
3.0       509
4.0        34
5.0     10210
6.0         6
99.0     1521
NaN     22825
Name: count, dtype: int64

Financial status:
Total amount > 0: 56683
Total amount = 0: 287
Total amount < 0: 3401

Duration status:
Duration > 0: 59613
Duration = 0: 758
Duration < 0: 0

Duration summary:


,duration_minutes
count,60371.000000
mean,10.192377
std,30.145658
min,0.000000
25%,0.216667
50%,5.900000
75%,15.283333
max,2957.466667



Sample zero-distance rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes,payment_type,RatecodeID,fare_amount,total_amount,PULocationID,DOLocationID
17,2,2024-01-01 00:52:09,2024-01-01 00:52:28,0.316667,2,1.0,3.0,8.00,237,237
23,1,2024-01-01 00:14:29,2024-01-01 00:14:29,0.000000,2,1.0,3.0,8.00,236,264
111,1,2024-01-01 00:58:50,2024-01-01 01:01:10,2.333333,1,1.0,4.4,11.40,162,162
198,1,2024-01-01 00:15:16,2024-01-01 00:26:58,11.700000,1,1.0,10.7,18.80,79,264
199,1,2024-01-01 00:39:34,2024-01-01 01:04:02,24.466667,1,1.0,19.8,26.80,161,264
251,1,2024-01-01 00:18:26,2024-01-01 00:52:40,34.233333,1,99.0,52.5,60.94,222,147
580,2,2024-01-01 00:04:05,2024-01-01 00:04:11,0.100000,1,5.0,12.2,13.20,143,143
593,1,2024-01-01 00:25:58,2024-01-01 00:27:19,1.350000,1,5.0,10.0,13.20,114,114
600,1,2024-01-01 00:31:10,2024-01-01 00:36:41,5.516667,2,1.0,6.5,11.50,264,162
709,2,2024-01-01 00:28:27,2024-01-01 00:29:15,0.800000,1,5.0,180.0,217.20,265,265


## 14. Non-positive trip durations

Inspect trips with zero or negative duration and determine which metrics they should be excluded from.


In [15]:
# Рассчитываем длительность для всей таблицы.
trips["duration_minutes"] = (
    trips["tpep_dropoff_datetime"]
    - trips["tpep_pickup_datetime"]
).dt.total_seconds() / 60

invalid_duration = trips[trips["duration_minutes"] <= 0].copy()

print("Rows with duration <= 0:", len(invalid_duration))

print("\nDuration breakdown:")
print("Duration = 0:", (invalid_duration["duration_minutes"] == 0).sum())
print("Duration < 0:", (invalid_duration["duration_minutes"] < 0).sum())

print("\nDistance status:")
print("Distance = 0:", (invalid_duration["trip_distance"] == 0).sum())
print("Distance > 0:", (invalid_duration["trip_distance"] > 0).sum())

print("\nFinancial status:")
print("Total amount > 0:", (invalid_duration["total_amount"] > 0).sum())
print("Total amount = 0:", (invalid_duration["total_amount"] == 0).sum())
print("Total amount < 0:", (invalid_duration["total_amount"] < 0).sum())

print("\nPayment types:")
print(
    invalid_duration["payment_type"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nDuration summary:")
display(
    invalid_duration["duration_minutes"]
    .describe()
    .to_frame(name="duration_minutes")
)

print("\nSample invalid-duration rows:")
display(
    invalid_duration[
        [
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "duration_minutes",
            "trip_distance",
            "payment_type",
            "RatecodeID",
            "fare_amount",
            "total_amount",
            "PULocationID",
            "DOLocationID"
        ]
    ].head(15)
)

Rows with duration <= 0: 870

Duration breakdown:
Duration = 0: 814
Duration < 0: 56

Distance status:
Distance = 0: 758
Distance > 0: 112

Financial status:
Total amount > 0: 816
Total amount = 0: 50
Total amount < 0: 4

Payment types:
payment_type
0    111
1    108
2    627
3     22
4      2
Name: count, dtype: int64

Duration summary:


,duration_minutes
count,870.000000
mean,-0.079464
std,0.758566
min,-13.566667
25%,0.000000
50%,0.000000
75%,0.000000
max,0.000000



Sample invalid-duration rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes,trip_distance,payment_type,RatecodeID,fare_amount,total_amount,PULocationID,DOLocationID
23,1,2024-01-01 00:14:29,2024-01-01 00:14:29,0.0,0.0,2,1.0,3.00,8.00,236,264
16790,1,2024-01-01 02:47:57,2024-01-01 02:47:57,0.0,0.0,1,1.0,52.50,54.00,61,61
17277,1,2024-01-01 03:18:03,2024-01-01 03:18:03,0.0,0.0,2,1.0,33.99,33.99,246,264
19195,1,2024-01-01 03:06:02,2024-01-01 03:06:02,0.0,0.0,2,1.0,35.22,35.22,113,264
21393,1,2024-01-01 04:08:24,2024-01-01 04:08:24,0.0,0.0,2,1.0,10.70,15.70,249,264
22868,1,2024-01-01 04:06:44,2024-01-01 04:06:44,0.0,0.0,2,1.0,14.51,14.51,137,264
23585,1,2024-01-01 05:21:13,2024-01-01 05:21:13,0.0,0.0,2,5.0,0.00,1.00,48,264
25800,1,2024-01-01 07:33:54,2024-01-01 07:33:54,0.0,0.0,1,99.0,66.70,66.70,223,264
25801,1,2024-01-01 07:40:28,2024-01-01 07:40:28,0.0,0.0,1,99.0,46.00,46.00,223,264
28009,1,2024-01-01 09:24:09,2024-01-01 09:24:09,0.0,0.0,2,1.0,3.00,7.00,238,264


## 15. Extreme-value diagnostics

Review unusually large distances, durations, fares, and total amounts. Diagnostic thresholds are not automatic deletion rules.


In [16]:
# Пороговые значения пока используем как диагностические,
# а не как окончательные правила удаления.

outlier_summary = pd.DataFrame({
    "Check": [
        "Trip distance > 100 miles",
        "Trip distance > 500 miles",
        "Duration > 180 minutes",
        "Duration > 1440 minutes",
        "Fare amount > 500",
        "Total amount > 500",
        "Total amount > 1000"
    ],
    "Rows": [
        (trips["trip_distance"] > 100).sum(),
        (trips["trip_distance"] > 500).sum(),
        (trips["duration_minutes"] > 180).sum(),
        (trips["duration_minutes"] > 1440).sum(),
        (trips["fare_amount"] > 500).sum(),
        (trips["total_amount"] > 500).sum(),
        (trips["total_amount"] > 1000).sum()
    ]
})

outlier_summary["Percent"] = (
    outlier_summary["Rows"] / len(trips) * 100
).round(4)

display(outlier_summary)

print("\nTop 10 longest distances:")
display(
    trips.nlargest(10, "trip_distance")[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "duration_minutes",
            "trip_distance",
            "fare_amount",
            "total_amount",
            "payment_type",
            "PULocationID",
            "DOLocationID"
        ]
    ]
)

print("\nTop 10 longest durations:")
display(
    trips.nlargest(10, "duration_minutes")[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "duration_minutes",
            "trip_distance",
            "fare_amount",
            "total_amount",
            "payment_type",
            "PULocationID",
            "DOLocationID"
        ]
    ]
)

print("\nTop 10 highest total amounts:")
display(
    trips.nlargest(10, "total_amount")[
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "duration_minutes",
            "trip_distance",
            "fare_amount",
            "tip_amount",
            "tolls_amount",
            "total_amount",
            "payment_type",
            "RatecodeID"
        ]
    ]
)

,Check,Rows,Percent
0,Trip distance > 100 miles,59,0.0020
1,Trip distance > 500 miles,25,0.0008
2,Duration > 180 minutes,1983,0.0669
3,Duration > 1440 minutes,16,0.0005
4,Fare amount > 500,46,0.0016
5,Total amount > 500,62,0.0021
6,Total amount > 1000,7,0.0002



Top 10 longest distances:


,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes,trip_distance,fare_amount,total_amount,payment_type,PULocationID,DOLocationID
2958079,2024-01-30 06:37:00,2024-01-30 06:50:00,13.0,312722.30,14.46,22.15,0,151,162
2934895,2024-01-25 08:39:00,2024-01-25 09:06:00,27.0,97793.92,29.71,36.31,0,4,13
2911232,2024-01-20 08:01:00,2024-01-20 08:26:00,25.0,82015.45,16.56,21.56,0,40,43
2920143,2024-01-21 11:58:00,2024-01-21 12:12:00,14.0,72975.97,12.70,20.04,0,209,211
2891991,2024-01-17 08:41:00,2024-01-17 09:23:00,42.0,71752.26,41.06,49.57,0,33,161
2843369,2024-01-05 15:46:00,2024-01-05 16:17:00,31.0,59282.45,32.02,33.52,0,74,47
2854947,2024-01-09 07:13:00,2024-01-09 07:17:00,4.0,59076.43,13.82,23.17,0,141,162
2931119,2024-01-24 12:26:00,2024-01-24 12:42:00,16.0,58298.51,12.94,18.63,0,151,236
2868605,2024-01-12 05:16:00,2024-01-12 05:26:00,10.0,51619.36,16.17,24.20,0,238,186
2869099,2024-01-12 07:18:00,2024-01-12 07:38:00,20.0,44018.64,32.75,52.43,0,233,138



Top 10 longest durations:


,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes,trip_distance,fare_amount,total_amount,payment_type,PULocationID,DOLocationID
2111505,2024-01-24 17:03:14,2024-01-31 06:38:38,9455.400000,2.26,30.3,36.80,2,237,170
942462,2024-01-12 01:20:50,2024-01-14 13:42:36,3621.766667,35.70,150.4,181.41,2,132,42
1159593,2024-01-14 10:08:11,2024-01-16 13:54:22,3106.183333,31.95,2221.3,2225.30,2,220,220
2758973,2024-01-31 12:39:24,2024-02-02 13:56:52,2957.466667,0.00,3.0,4.50,2,207,260
1393159,2024-01-17 05:31:17,2024-01-19 05:12:10,2860.883333,0.20,28.9,33.90,2,162,162
2078351,2024-01-24 12:39:56,2024-01-26 07:45:44,2585.800000,0.00,3.0,4.50,2,207,226
2219253,2024-01-25 18:53:29,2024-01-27 11:32:13,2438.733333,4.56,21.9,28.40,2,140,133
837565,2024-01-10 23:28:44,2024-01-12 11:02:04,2133.333333,37.97,-70.0,-73.25,4,132,132
837566,2024-01-10 23:28:44,2024-01-12 11:02:04,2133.333333,37.97,70.0,73.25,4,132,132
1871822,2024-01-22 06:29:10,2024-01-23 17:07:01,2077.850000,20.35,70.0,75.75,2,132,100



Top 10 highest total amounts:


,tpep_pickup_datetime,tpep_dropoff_datetime,duration_minutes,trip_distance,fare_amount,tip_amount,tolls_amount,total_amount,payment_type,RatecodeID
1714869,2024-01-20 11:18:47,2024-01-20 11:18:47,0.000000,0.00,5000.0,0.0,0.00,5000.00,3,99.0
1714870,2024-01-20 11:19:33,2024-01-20 11:19:33,0.000000,0.00,5000.0,0.0,0.00,5000.00,3,99.0
1714871,2024-01-20 11:20:15,2024-01-20 11:20:15,0.000000,0.00,2500.0,0.0,0.00,2500.00,3,99.0
1714873,2024-01-20 11:27:48,2024-01-20 11:27:48,0.000000,0.00,2500.0,0.0,0.00,2500.00,1,99.0
2084560,2024-01-24 13:44:43,2024-01-24 13:44:43,0.000000,0.00,2500.0,0.0,0.00,2500.00,1,99.0
1159593,2024-01-14 10:08:11,2024-01-16 13:54:22,3106.183333,31.95,2221.3,0.0,0.00,2225.30,2,1.0
76891,2024-01-02 07:50:08,2024-01-02 11:29:29,219.350000,233.25,1616.5,0.0,0.00,1617.50,2,4.0
1714872,2024-01-20 11:22:27,2024-01-20 11:22:27,0.000000,0.00,1000.0,0.0,0.00,1000.00,3,99.0
507314,2024-01-06 21:01:38,2024-01-06 23:41:44,160.100000,142.62,912.3,0.0,26.63,940.93,2,4.0
1916205,2024-01-22 16:40:43,2024-01-22 19:11:20,150.616667,157.25,899.0,0.0,0.00,900.00,2,5.0


## 16. Duplicate checks

Check both exact duplicate rows and duplicate combinations of core trip attributes.


In [17]:
# 1. Полностью одинаковые строки по всем колонкам
full_duplicates = trips.duplicated(keep=False)

print("Rows that are part of full duplicate groups:")
print(full_duplicates.sum())

print("\nNumber of duplicate rows beyond the first copy:")
print(trips.duplicated().sum())


# 2. Потенциальные дубликаты по основным признакам поездки
trip_key_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "total_amount"
]

potential_duplicates = trips.duplicated(
    subset=trip_key_columns,
    keep=False
)

print("\nRows that share the same trip-key values:")
print(potential_duplicates.sum())


# Показываем примеры полных дубликатов
print("\nSample full duplicates:")
display(
    trips.loc[full_duplicates]
    .sort_values(
        [
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID",
            "DOLocationID"
        ]
    )
    .head(20)
)


# Показываем примеры потенциальных дубликатов
print("\nSample potential duplicates:")
display(
    trips.loc[
        potential_duplicates,
        trip_key_columns + [
            "payment_type",
            "RatecodeID",
            "tip_amount"
        ]
    ]
    .sort_values(trip_key_columns)
    .head(20)
)

Rows that are part of full duplicate groups:
0

Number of duplicate rows beyond the first copy:
0

Rows that share the same trip-key values:
0

Sample full duplicates:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration_minutes



Sample potential duplicates:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount,payment_type,RatecodeID,tip_amount


## 17. Taxi-zone lookup integrity

Verify that `LocationID` is unique and that every pickup and drop-off ID used in the trip table exists in the zone lookup.


In [18]:
# Проверяем справочник taxi zones.

print("Rows in zones table:", len(zones))
print("Unique LocationID values:", zones["LocationID"].nunique())

print(
    "Duplicate LocationID rows:",
    zones["LocationID"].duplicated().sum()
)

print("\nMissing values in zones table:")
print(zones.isna().sum())


# Множество всех LocationID из справочника.
valid_zone_ids = set(zones["LocationID"])

# ID, которые используются в trips, но отсутствуют в справочнике.
missing_pickup_ids = sorted(
    set(trips["PULocationID"]) - valid_zone_ids
)

missing_dropoff_ids = sorted(
    set(trips["DOLocationID"]) - valid_zone_ids
)

print("\nPickup LocationIDs missing from zones lookup:")
print(missing_pickup_ids)

print("\nNumber of trip rows with missing pickup zone:")
print(
    (~trips["PULocationID"].isin(valid_zone_ids)).sum()
)

print("\nDrop-off LocationIDs missing from zones lookup:")
print(missing_dropoff_ids)

print("\nNumber of trip rows with missing drop-off zone:")
print(
    (~trips["DOLocationID"].isin(valid_zone_ids)).sum()
)

Rows in zones table: 265
Unique LocationID values: 265
Duplicate LocationID rows: 0

Missing values in zones table:
LocationID      0
Borough         1
Zone            1
service_zone    2
dtype: int64

Pickup LocationIDs missing from zones lookup:
[]

Number of trip rows with missing pickup zone:
0

Drop-off LocationIDs missing from zones lookup:
[]

Number of trip rows with missing drop-off zone:
0


## 18. Data-quality conclusions

### Main findings

- The trip table contains **2,964,624 records** and 19 original source columns.
- No exact duplicate rows or duplicate trip-key combinations were found.
- **140,162 records (4.73%)** share the same five missing fields and match `payment_type = 0` exactly.
- All pickup and drop-off location IDs are covered by the taxi-zone lookup.
- Zero-distance and non-positive-duration records are not removed globally because some still contain usable zone, time, or financial information.
- Negative monetary records are kept in the raw data but excluded from standard positive-revenue KPIs.
- Clearly impossible distance and duration outliers are excluded only from metrics they would distort.

### Analytical treatment rules

| Analysis type | Treatment rule |
|---|---|
| January trip counts | Require pickup timestamp within January 2024 |
| Passenger analysis | Require non-missing `passenger_count` |
| Rate-code analysis | Require non-missing `RatecodeID` |
| Distance metrics | Require positive, plausible `trip_distance` |
| Duration metrics | Require positive, plausible `duration_minutes` |
| Standard revenue KPIs | Require `total_amount > 0` |
| Revenue per mile | Require valid distance and positive total amount |
| Revenue per minute | Require valid duration and positive total amount |
| Payment analysis | Keep `payment_type = 0` as a separate category |
| Zone analysis | All pickup and drop-off IDs can be joined to the zone lookup |

The raw source records remain unchanged. Metric-specific filters will be implemented during the PostgreSQL analysis.
